In [11]:
pip install --upgrade --force-reinstall transformers

  Using cached transformers-4.56.0-py3-none-any.whl.metadata (40 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.3.2-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached PyYAML-6.0.2-cp313-cp313-win_amd64.whl.metadata (2.1 kB)
  Using cached regex-2025.8.29-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.22.0-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached charset_norm

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.7.0 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install --upgrade accelerate

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd
import numpy as np

In [ ]:
import transformers
print(transformers.__version__)

4.55.4


In [ ]:

df = pd.read_csv(r'..\data\processed\full_data.csv')
df.head

<bound method NDFrame.head of                                                      text   emotion
0       @tiffanylue i know  i was listenin to bad habi...       sad
1       Layin n bed with a headache  ughhhh...waitin o...       sad
2                     Funeral ceremony...gloomy friday...       sad
3                    wants to hang out with friends SOON!       joy
4       Re-pinging @ghostridah14: why didn't you go to...      fear
...                                                   ...       ...
496217  I was born without a little finger on my right...  surprise
496218                             "Not OJ" license plate  surprise
496219             I welded a rose for my mum's birthday.  surprise
496220  These two different sets of shopping baskets a...  surprise
496221            These removed fish hooks at my local ER  surprise

[496222 rows x 2 columns]>

In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["emotion"])


train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


In [ ]:
from transformers import AutoTokenizer

model_name = "roberta-base"  
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map:   0%|          | 0/446599 [00:00<?, ? examples/s]

Map:   0%|          | 0/49623 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = 5  # fear, sad, anger, surprise, joy
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # ajusta según GPU
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)


C:\Users\nahue\AppData\Local\Temp\ipykernel_15012\284246441.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


: 

In [ ]:
trainer.train()


c:\Users\nahue\Documents\sp-ml-17-final-project-g1\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,1.260700
200,0.785300
300,0.638800
400,0.541300
500,0.510900
600,0.445000
700,0.450000
800,0.441700
900,0.416900
1000,0.410800
